# Adaptive CoRT-SI Kaggle Notebook

This notebook inlines the full implementation from the local `cort_si` package so you can upload a single `.ipynb` file to Kaggle and run it without needing the Python package files.

The code below covers the logic from these repo modules:
- `gen_data.py`
- `utils.py`
- `transfer_learning_hdr.py`
- `sub_prob.py`
- `CORT_SI.py`

In [1]:
import importlib.util
import subprocess
import sys

required = {
    "numpy": "numpy",
    "scipy": "scipy",
    "mpmath": "mpmath",
    "skglm": "skglm",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    print("Installed:", ", ".join(missing))
else:
    print("All required packages are already available.")

All required packages are already available.


In [2]:
import random
import warnings
from types import SimpleNamespace

import numpy as np
from numpy.linalg import pinv
from scipy.linalg import block_diag
from mpmath import mp
from skglm import Lasso, WeightedLasso

warnings.filterwarnings("ignore")
mp.dps = 500

In [3]:
def generate_coef(p, s, true_beta=0.25, num_info_aux=3, num_uninfo_aux=2, gamma=0.01):
    K = num_info_aux + num_uninfo_aux
    beta_0 = np.concatenate([np.full(s, true_beta), np.zeros(p - s)])
    noisy_prefix = min(p, 50)
    informative_prefix = min(p, 25)

    Beta_S = np.tile(beta_0, (K, 1)).T
    if s >= 0:
        Beta_S[0, :] -= 2 * true_beta
        for m in range(K):
            if m < num_uninfo_aux:
                Beta_S[:noisy_prefix, m] += np.random.normal(0, true_beta * gamma * 10, noisy_prefix)
            else:
                Beta_S[:informative_prefix, m] += np.random.normal(0, true_beta * gamma, informative_prefix)
    return Beta_S, beta_0


def generate_data(p, s, nS, nT, true_beta=0.25, num_info_aux=3, num_uninfo_aux=2, gamma=0.01):
    K = num_info_aux + num_uninfo_aux

    Beta_S, beta_0 = generate_coef(p, s, true_beta, num_info_aux, num_uninfo_aux, gamma)
    Beta = np.column_stack([Beta_S[:, i] for i in range(K)] + [beta_0])

    X_list = []
    Y_list = []
    true_Y_list = []

    cov = np.eye(p)
    N_vec = [nS] * K + [nT]

    for k in range(K + 1):
        Xk = np.random.multivariate_normal(mean=np.zeros(p), cov=cov, size=N_vec[k])
        true_Yk = Xk @ Beta[:, k]
        noise = np.random.normal(0, 1, N_vec[k])
        Yk = true_Yk + noise
        X_list.append(Xk)
        Y_list.append(Yk)
        true_Y_list.append(true_Yk)

    XS_list = X_list[:-1]
    YS_list = Y_list[:-1]
    X0 = X_list[-1]
    Y0 = Y_list[-1]
    true_Y = np.concatenate(true_Y_list)
    SigmaS_list = [np.eye(nS) for _ in range(K)]
    Sigma0 = np.eye(nT)

    return XS_list, YS_list, X0, Y0, true_Y, SigmaS_list, Sigma0, beta_0


gen_data = SimpleNamespace(
    generate_coef=generate_coef,
    generate_data=generate_data,
)


def construct_Sigma(SigmaS_list, Sigma0):
    Sigma = block_diag(*SigmaS_list, Sigma0)
    return Sigma


def construct_betaM_M_SM_Mc(beta_hat):
    M = []
    betaM = []
    SM = []
    Mc = []

    for i, val in enumerate(beta_hat):
        if val != 0.0:
            M.append(i)
            SM.append(np.sign(val))
            betaM.append(val)
        else:
            Mc.append(i)

    SM = np.array(SM).reshape(-1, 1)

    return betaM, M, SM, Mc


def construct_test_statistic(j, X0M, Y, M, nT, N):
    ej = np.zeros(len(M))

    for i, ac in enumerate(M):
        if ac == j:
            ej[i] = 1
            break

    inv = np.linalg.pinv(X0M.T @ X0M)
    X0M_inv = X0M @ inv

    _X = np.zeros((N, len(M)))
    _X[N - nT :, :] = X0M_inv
    etaj = _X @ ej
    etajTY = float(etaj @ Y)

    return etaj.reshape(-1, 1), etajTY


def calculate_a_b(etaj, Y, Sigma, N):
    e1 = etaj.T @ Sigma @ etaj
    b = (Sigma @ etaj) / e1

    e2 = np.eye(N) - b @ etaj.T
    a = e2 @ Y

    return a.reshape(-1, 1), b.reshape(-1, 1)


def merge_intervals(intervals, tol=1e-3):
    intervals = sorted(intervals, key=lambda x: x[0])
    merged = []
    for interval in intervals:
        if not merged or interval[0] - merged[-1][1] > tol:
            merged.append(interval)
        else:
            merged[-1] = (merged[-1][0], max(merged[-1][1], interval[1]))
    return merged


def pivot(intervals, etajTy, etaj, Sigma, tn_mu=0):
    if len(intervals) == 0:
        return None
    intervals = merge_intervals(intervals, tol=1e-3)

    etaj = etaj.ravel()
    stdev = np.sqrt(etaj @ (Sigma @ etaj))

    numerator = mp.mpf("0")
    denominator = mp.mpf("0")

    for left, right in intervals:
        cdf_left = mp.ncdf((left - tn_mu) / stdev)
        cdf_right = mp.ncdf((right - tn_mu) / stdev)
        piece = cdf_right - cdf_left
        denominator += piece

        if etajTy >= right:
            numerator += piece
        elif left <= etajTy < right:
            numerator += mp.ncdf((etajTy - tn_mu) / stdev) - cdf_left

    if denominator == 0:
        return None
    return float(numerator / denominator)


def calculate_TN_p_value(intervals, etaj, etajTy, Sigma, tn_mu=0.0):
    cdf = pivot(intervals, etajTy, etaj, Sigma, tn_mu)
    return 2.0 * min(cdf, 1.0 - cdf)


def construct_Y(YS_list, Y0):
    return np.concatenate([np.asarray(y).ravel() for y in YS_list] + [np.asarray(Y0).ravel()])


def construct_Y_tilde(YS_list, Y0, source_set):
    blocks = [(np.asarray(YS_list[source_idx]).ravel() / np.sqrt(YS_list[source_idx].shape[0])) for source_idx in source_set]
    blocks.append(np.asarray(Y0).ravel() / np.sqrt(Y0.shape[0]))
    return np.concatenate(blocks)


def construct_X_tilde(XS_list, X0, source_set):
    p = X0.shape[1]
    num_sources = len(source_set)
    rows = []

    for block_idx, source_idx in enumerate(source_set):
        Xk = XS_list[source_idx]
        nk = Xk.shape[0]
        Xk_scaled = Xk / np.sqrt(nk)
        row = [Xk_scaled if idx == block_idx else np.zeros((nk, p)) for idx in range(num_sources)]
        row.append(Xk_scaled)
        rows.append(np.hstack(row))

    n0 = X0.shape[0]
    row0 = [np.zeros((n0, p)) for _ in range(num_sources)]
    row0.append(X0 / np.sqrt(n0))
    rows.append(np.hstack(row0))
    return np.vstack(rows)


def construct_w_tilde(p, lambda0, lambdak_list, source_set):
    weights = [np.full(p, lambdak_list[source_idx]) for source_idx in source_set]
    weights.append(np.full(p, lambda0))
    return np.concatenate(weights)


def construct_folds(n0, T=5, shuffle=False, random_state=0):
    indices = np.arange(n0)
    if shuffle:
        rng = np.random.default_rng(random_state)
        indices = rng.permutation(indices)
    return [np.array(split, dtype=int) for split in np.array_split(indices, T)]


def construct_block_slices(ns_list, n0):
    slices = []
    start = 0
    for nk in ns_list:
        stop = start + nk
        slices.append(slice(start, stop))
        start = stop
    slices.append(slice(start, start + n0))
    return slices


def split_stacked_response(y, ns_list, n0):
    y = np.asarray(y)
    block_slices = construct_block_slices(ns_list, n0)
    source_blocks = [np.asarray(y[block_slices[idx]]).reshape(-1) for idx in range(len(ns_list))]
    target_block = np.asarray(y[block_slices[-1]]).reshape(-1)
    return source_blocks, target_block


def complement_fold_indices(n0, fold_indices):
    mask = np.ones(n0, dtype=bool)
    mask[np.asarray(fold_indices, dtype=int)] = False
    return np.flatnonzero(mask)


def solve_linear_inequalities_1d(psi, gamma, tol=1e-12):
    psi = np.asarray(psi, dtype=float).ravel()
    gamma = np.asarray(gamma, dtype=float).ravel()

    left = -np.inf
    right = np.inf

    for coeff, bound in zip(psi, gamma):
        if abs(coeff) <= tol:
            if bound < -tol:
                return []
            continue

        value = bound / coeff
        if coeff > 0:
            right = min(right, value)
        else:
            left = max(left, value)

    if right + tol < left:
        return []
    return [(left, right)]


def solve_quadratic_leq(A, B, C, tol=1e-12):
    A = float(A)
    B = float(B)
    C = float(C)

    if abs(A) <= tol:
        return solve_linear_inequalities_1d([B], [-C], tol=tol)

    discriminant = (B * B) - (4.0 * A * C)
    if discriminant < -tol:
        return [(-np.inf, np.inf)] if A < 0 else []

    if abs(discriminant) <= tol:
        root = -B / (2.0 * A)
        return [(-np.inf, np.inf)] if A < 0 else [(root, root)]

    sqrt_discriminant = np.sqrt(discriminant)
    root1 = (-B - sqrt_discriminant) / (2.0 * A)
    root2 = (-B + sqrt_discriminant) / (2.0 * A)
    left, right = min(root1, root2), max(root1, root2)

    if A > 0:
        return [(left, right)]
    return [(-np.inf, left), (right, np.inf)]


def intersect_interval_unions(intervals_a, intervals_b, tol=1e-8):
    if not intervals_a or not intervals_b:
        return []

    intervals_a = merge_intervals(list(intervals_a), tol=tol)
    intervals_b = merge_intervals(list(intervals_b), tol=tol)

    result = []
    idx_a = 0
    idx_b = 0

    while idx_a < len(intervals_a) and idx_b < len(intervals_b):
        left = max(intervals_a[idx_a][0], intervals_b[idx_b][0])
        right = min(intervals_a[idx_a][1], intervals_b[idx_b][1])
        if left <= right + tol:
            result.append((left, right))

        if intervals_a[idx_a][1] < intervals_b[idx_b][1] - tol:
            idx_a += 1
        else:
            idx_b += 1

    return merge_intervals(result, tol=tol)


def union_interval_unions(intervals_a, intervals_b, tol=1e-8):
    if not intervals_a:
        return merge_intervals(list(intervals_b), tol=tol) if intervals_b else []
    if not intervals_b:
        return merge_intervals(list(intervals_a), tol=tol)
    return merge_intervals(list(intervals_a) + list(intervals_b), tol=tol)


def clip_interval_union(intervals, left, right, tol=1e-8):
    return intersect_interval_unions(intervals, [(left, right)], tol=tol)


def point_in_interval_union(value, intervals, tol=1e-10):
    for left, right in intervals:
        if left - tol <= value <= right + tol:
            return True
    return False


def count_region_from_fold_wins(win_regions, m, mode, z_min=None, z_max=None, tol=1e-8):
    if mode not in {"selected", "discarded"}:
        raise ValueError("mode must be 'selected' or 'discarded'")

    endpoints = []
    if z_min is not None and z_max is not None:
        endpoints.extend([float(z_min), float(z_max)])

    for intervals in win_regions:
        for left, right in intervals:
            if np.isfinite(left):
                endpoints.append(float(left))
            if np.isfinite(right):
                endpoints.append(float(right))

    if not endpoints:
        return [] if mode == "selected" else [(-np.inf, np.inf)]

    endpoints = sorted(set(endpoints))
    if len(endpoints) == 1:
        return [(endpoints[0], endpoints[0])] if mode == "discarded" else []

    kept = []
    for left, right in zip(endpoints[:-1], endpoints[1:]):
        if right < left + tol:
            continue
        midpoint = 0.5 * (left + right)
        wins = sum(point_in_interval_union(midpoint, intervals, tol=tol) for intervals in win_regions)

        if mode == "selected" and wins >= m:
            kept.append((left, right))
        if mode == "discarded" and wins <= m - 1:
            kept.append((left, right))

    return merge_intervals(kept, tol=tol)


utils = SimpleNamespace(
    construct_Sigma=construct_Sigma,
    construct_betaM_M_SM_Mc=construct_betaM_M_SM_Mc,
    construct_test_statistic=construct_test_statistic,
    calculate_a_b=calculate_a_b,
    merge_intervals=merge_intervals,
    pivot=pivot,
    calculate_TN_p_value=calculate_TN_p_value,
    construct_Y=construct_Y,
    construct_Y_tilde=construct_Y_tilde,
    construct_X_tilde=construct_X_tilde,
    construct_w_tilde=construct_w_tilde,
    construct_folds=construct_folds,
    construct_block_slices=construct_block_slices,
    split_stacked_response=split_stacked_response,
    complement_fold_indices=complement_fold_indices,
    solve_linear_inequalities_1d=solve_linear_inequalities_1d,
    solve_quadratic_leq=solve_quadratic_leq,
    intersect_interval_unions=intersect_interval_unions,
    union_interval_unions=union_interval_unions,
    clip_interval_union=clip_interval_union,
    point_in_interval_union=point_in_interval_union,
    count_region_from_fold_wins=count_region_from_fold_wins,
)


def solve_lasso(X, y, lam, fit_intercept=False, tol=1e-10, max_iter=10_000):
    model = Lasso(alpha=lam, fit_intercept=fit_intercept, tol=tol, max_iter=max_iter)
    model.fit(X, np.asarray(y).ravel())
    return model.coef_


def solve_cort_model(X0, Y0, XS_list, YS_list, source_set, lambda0, lambdak_list, tol=1e-10):
    X_tilde = utils.construct_X_tilde(XS_list, X0, source_set)
    Y_tilde = utils.construct_Y_tilde(YS_list, Y0, source_set)
    w_tilde = utils.construct_w_tilde(X0.shape[1], lambda0, lambdak_list, source_set)

    weighted_lasso = WeightedLasso(
        alpha=1.0 / X_tilde.shape[0],
        fit_intercept=False,
        tol=tol,
        weights=w_tilde,
    )
    weighted_lasso.fit(X_tilde, Y_tilde)
    theta_hat = weighted_lasso.coef_
    beta0_hat = theta_hat[-X0.shape[1]:]
    return theta_hat, beta0_hat, X_tilde, w_tilde


def adaptive_source_selection(X0, Y0, XS_list, YS_list, folds, lambda_sel):
    n0 = Y0.shape[0]
    if folds is None:
        folds = utils.construct_folds(n0, T=5, shuffle=False)

    if len(folds) % 2 == 0:
        raise ValueError("The number of folds T must be odd for majority voting")

    selected_sources = []
    majority = (len(folds) + 1) // 2

    for source_idx, (Xk, Yk) in enumerate(zip(XS_list, YS_list)):
        source_votes = []

        for fold_indices in folds:
            train_indices = utils.complement_fold_indices(n0, fold_indices)

            X0_train = X0[train_indices]
            Y0_train = Y0[train_indices]
            X0_valid = X0[fold_indices]
            Y0_valid = Y0[fold_indices]

            beta_target = solve_lasso(X0_train, Y0_train, lambda_sel)
            X_aug_train = np.vstack([Xk, X0_train])
            Y_aug_train = np.concatenate([Yk, Y0_train])
            beta_aug = solve_lasso(X_aug_train, Y_aug_train, lambda_sel)

            loss_target = 0.5 * np.mean((Y0_valid - X0_valid @ beta_target) ** 2)
            loss_aug = 0.5 * np.mean((Y0_valid - X0_valid @ beta_aug) ** 2)
            source_votes.append(loss_aug <= loss_target)

        if int(np.sum(source_votes)) >= majority:
            selected_sources.append(source_idx)

    return selected_sources


transfer_learning_hdr = SimpleNamespace(
    solve_lasso=solve_lasso,
    solve_cort_model=solve_cort_model,
    adaptive_source_selection=adaptive_source_selection,
)

In [4]:
def _active_set_from_coef(coef):
    coef = np.asarray(coef).ravel()
    return [idx for idx, value in enumerate(coef) if value != 0]


def solve_lasso(X, y, lam, fit_intercept=False, tol=1e-10, max_iter=10_000, verbose=False, label=None):
    model = Lasso(alpha=lam, fit_intercept=fit_intercept, tol=tol, max_iter=max_iter)
    model.fit(X, np.asarray(y).ravel())
    coef = model.coef_
    if verbose:
        prefix = f"{label}: " if label else ""
        print(f"{prefix}active set = {_active_set_from_coef(coef)}")
    return coef


def solve_cort_model(X0, Y0, XS_list, YS_list, source_set, lambda0, lambdak_list, tol=1e-10, verbose=False, label="CoRT"):
    X_tilde = utils.construct_X_tilde(XS_list, X0, source_set)
    Y_tilde = utils.construct_Y_tilde(YS_list, Y0, source_set)
    w_tilde = utils.construct_w_tilde(X0.shape[1], lambda0, lambdak_list, source_set)

    weighted_lasso = WeightedLasso(
        alpha=1.0 / X_tilde.shape[0],
        fit_intercept=False,
        tol=tol,
        weights=w_tilde,
    )
    weighted_lasso.fit(X_tilde, Y_tilde)
    theta_hat = weighted_lasso.coef_
    beta0_hat = theta_hat[-X0.shape[1]:]
    if verbose:
        print(f"{label}: source set = {list(source_set)}")
        print(f"{label}: combined active set = {_active_set_from_coef(theta_hat)}")
        print(f"{label}: target active set = {_active_set_from_coef(beta0_hat)}")
    return theta_hat, beta0_hat, X_tilde, w_tilde


def adaptive_source_selection(X0, Y0, XS_list, YS_list, folds, lambda_sel, verbose=False):
    n0 = Y0.shape[0]
    if folds is None:
        folds = utils.construct_folds(n0, T=5, shuffle=False)

    if len(folds) % 2 == 0:
        raise ValueError("The number of folds T must be odd for majority voting")

    selected_sources = []
    majority = (len(folds) + 1) // 2

    for source_idx, (Xk, Yk) in enumerate(zip(XS_list, YS_list)):
        source_votes = []

        for fold_idx, fold_indices in enumerate(folds):
            train_indices = utils.complement_fold_indices(n0, fold_indices)

            X0_train = X0[train_indices]
            Y0_train = Y0[train_indices]
            X0_valid = X0[fold_indices]
            Y0_valid = Y0[fold_indices]

            fold_tag = f"source {source_idx}, fold {fold_idx + 1}"
            beta_target = solve_lasso(
                X0_train,
                Y0_train,
                lambda_sel,
                verbose=verbose,
                label=f"{fold_tag} target-only",
            )
            X_aug_train = np.vstack([Xk, X0_train])
            Y_aug_train = np.concatenate([Yk, Y0_train])
            beta_aug = solve_lasso(
                X_aug_train,
                Y_aug_train,
                lambda_sel,
                verbose=verbose,
                label=f"{fold_tag} target+source",
            )

            loss_target = 0.5 * np.mean((Y0_valid - X0_valid @ beta_target) ** 2)
            loss_aug = 0.5 * np.mean((Y0_valid - X0_valid @ beta_aug) ** 2)
            vote = loss_aug <= loss_target
            source_votes.append(vote)
            if verbose:
                print(f"{fold_tag}: vote = {vote}")

        num_votes = int(np.sum(source_votes))
        if verbose:
            print(f"source {source_idx}: votes = {num_votes}/{len(folds)}")

        if num_votes >= majority:
            selected_sources.append(source_idx)

    if verbose:
        print(f"selected source set = {selected_sources}")

    return selected_sources


transfer_learning_hdr.solve_lasso = solve_lasso
transfer_learning_hdr.solve_cort_model = solve_cort_model
transfer_learning_hdr.adaptive_source_selection = adaptive_source_selection

In [5]:
def lasso_state_interval(X_train, a_train, b_train, active_set, sign_vec, lambda_sel, n_samples):
    a_train = np.asarray(a_train, dtype=float).reshape(-1, 1)
    b_train = np.asarray(b_train, dtype=float).reshape(-1, 1)
    active_set = list(active_set)
    inactive_set = [idx for idx in range(X_train.shape[1]) if idx not in active_set]
    sign_vec = np.asarray(sign_vec, dtype=float).reshape(-1, 1)

    c_full = np.zeros((X_train.shape[1], 1))
    d_full = np.zeros((X_train.shape[1], 1))
    psi0 = np.array([])
    gamma0 = np.array([])
    psi1 = np.array([])
    gamma1 = np.array([])

    if active_set:
        X_active = X_train[:, active_set]
        gram_inv = pinv(X_active.T @ X_active)
        X_active_plus = gram_inv @ X_active.T
        embed_active = np.eye(X_train.shape[1])[:, active_set]

        c_active = gram_inv @ (X_active.T @ a_train - (n_samples * lambda_sel) * sign_vec)
        d_active = gram_inv @ (X_active.T @ b_train)
        c_full = embed_active @ c_active
        d_full = embed_active @ d_active

        psi0 = (-sign_vec * (X_active_plus @ b_train)).ravel()
        gamma0 = (sign_vec * (X_active_plus @ a_train - (n_samples * lambda_sel) * (gram_inv @ sign_vec))).ravel()
        projection = np.eye(X_train.shape[0]) - X_active @ X_active_plus
    else:
        X_active = np.zeros((X_train.shape[0], 0))
        gram_inv = np.zeros((0, 0))
        projection = np.eye(X_train.shape[0])

    if inactive_set:
        X_inactive = X_train[:, inactive_set]
        inactive_term = X_inactive.T @ projection
        term_b = inactive_term @ b_train
        psi1 = np.concatenate([
            (term_b / (n_samples * lambda_sel)).ravel(),
            (-term_b / (n_samples * lambda_sel)).ravel(),
        ])

        if active_set:
            coupling = X_inactive.T @ X_active @ gram_inv @ sign_vec
        else:
            coupling = np.zeros((len(inactive_set), 1))

        term_a = inactive_term @ a_train
        gamma1 = np.concatenate([
            (np.ones_like(term_a) - coupling - (term_a / (n_samples * lambda_sel))).ravel(),
            (np.ones_like(term_a) + coupling + (term_a / (n_samples * lambda_sel))).ravel(),
        ])

    psi = np.concatenate((psi0, psi1))
    gamma = np.concatenate((gamma0, gamma1))
    interval = utils.solve_linear_inequalities_1d(psi, gamma)
    return psi, gamma, c_full, d_full, interval


def target_fold_state_interval(X0_train, a0_train, b0_train, active_set, sign_vec, lambda_sel, n_train):
    return lasso_state_interval(X0_train, a0_train, b0_train, active_set, sign_vec, lambda_sel, n_train)


def augmented_fold_state_interval(X_aug_train, a_aug_train, b_aug_train, active_set, sign_vec, lambda_sel, n_aug):
    return lasso_state_interval(X_aug_train, a_aug_train, b_aug_train, active_set, sign_vec, lambda_sel, n_aug)


def validation_quadratic(X0_val, a0_val, b0_val, c0, d0, ck, dk, n_val):
    a0_val = np.asarray(a0_val, dtype=float).reshape(-1, 1)
    b0_val = np.asarray(b0_val, dtype=float).reshape(-1, 1)
    c0 = np.asarray(c0, dtype=float).reshape(-1, 1)
    d0 = np.asarray(d0, dtype=float).reshape(-1, 1)
    ck = np.asarray(ck, dtype=float).reshape(-1, 1)
    dk = np.asarray(dk, dtype=float).reshape(-1, 1)

    residual0_a = a0_val - X0_val @ c0
    residual0_b = b0_val - X0_val @ d0
    residualk_a = a0_val - X0_val @ ck
    residualk_b = b0_val - X0_val @ dk

    A0 = 0.5 * float(residual0_b.T @ residual0_b) / n_val
    B0 = float(residual0_a.T @ residual0_b) / n_val
    C0 = 0.5 * float(residual0_a.T @ residual0_a) / n_val

    Ak = 0.5 * float(residualk_b.T @ residualk_b) / n_val
    Bk = float(residualk_a.T @ residualk_b) / n_val
    Ck = 0.5 * float(residualk_a.T @ residualk_a) / n_val

    return Ak - A0, Bk - B0, Ck - C0


def kkt_interval(X_tilde, a_adapt, b_adapt, theta_hat, w_tilde, p, tol=1e-10):
    _ = tol
    a_adapt = np.asarray(a_adapt, dtype=float).reshape(-1, 1)
    b_adapt = np.asarray(b_adapt, dtype=float).reshape(-1, 1)
    theta_hat = np.asarray(theta_hat, dtype=float).ravel()
    w_tilde = np.asarray(w_tilde, dtype=float).reshape(-1, 1)

    active_set = [idx for idx, value in enumerate(theta_hat) if value != 0]
    inactive_set = [idx for idx in range(X_tilde.shape[1]) if idx not in active_set]
    sign_vec = np.sign(theta_hat[active_set]).reshape(-1, 1) if active_set else np.zeros((0, 1))

    psi0 = np.array([])
    gamma0 = np.array([])
    psi1 = np.array([])
    gamma1 = np.array([])

    if active_set:
        X_active = X_tilde[:, active_set]
        gram_inv = pinv(X_active.T @ X_active)
        X_active_plus = gram_inv @ X_active.T
        weighted_sign = w_tilde[active_set] * sign_vec

        psi0 = (-sign_vec * (X_active_plus @ b_adapt)).ravel()
        gamma0 = (sign_vec * (X_active_plus @ a_adapt) - sign_vec * (gram_inv @ weighted_sign)).ravel()
        projection = np.eye(X_tilde.shape[0]) - X_active @ X_active_plus
    else:
        X_active = np.zeros((X_tilde.shape[0], 0))
        gram_inv = np.zeros((0, 0))
        weighted_sign = np.zeros((0, 1))
        projection = np.eye(X_tilde.shape[0])

    if inactive_set:
        X_inactive = X_tilde[:, inactive_set]
        inactive_term = X_inactive.T @ projection
        term_b = inactive_term @ b_adapt
        psi1 = np.concatenate([term_b.ravel(), -term_b.ravel()])

        if active_set:
            coupling = X_inactive.T @ X_active @ gram_inv @ weighted_sign
        else:
            coupling = np.zeros((len(inactive_set), 1))

        term_a = inactive_term @ a_adapt
        gamma1 = np.concatenate([
            (w_tilde[inactive_set] - coupling - term_a).ravel(),
            (w_tilde[inactive_set] + coupling + term_a).ravel(),
        ])

    psi = np.concatenate((psi0, psi1))
    gamma = np.concatenate((gamma0, gamma1))
    interval = utils.solve_linear_inequalities_1d(psi, gamma)

    target_start = X_tilde.shape[1] - p
    target_active = [idx - target_start for idx in active_set if idx >= target_start]
    return interval, target_active


def fold_win_region(source_idx, fold_idx, X0, Y0, XS_list, YS_list, a, b, folds, lambda_sel, z_min, z_max, eps=1e-5, tol=1e-10):
    ns_list = [ys.shape[0] for ys in YS_list]
    source_a_blocks, target_a = utils.split_stacked_response(a, ns_list, Y0.shape[0])
    source_b_blocks, target_b = utils.split_stacked_response(b, ns_list, Y0.shape[0])

    fold_indices = np.asarray(folds[fold_idx], dtype=int)
    train_indices = utils.complement_fold_indices(Y0.shape[0], fold_indices)
    X0_train = X0[train_indices]
    X0_val = X0[fold_indices]

    a0_train = target_a[train_indices]
    b0_train = target_b[train_indices]
    a0_val = target_a[fold_indices]
    b0_val = target_b[fold_indices]

    X_source = XS_list[source_idx]
    a_source = source_a_blocks[source_idx]
    b_source = source_b_blocks[source_idx]
    X_aug_train = np.vstack([X_source, X0_train])
    a_aug_train = np.concatenate([a_source, a0_train])
    b_aug_train = np.concatenate([b_source, b0_train])

    intervals = []
    z = z_min
    while z < z_max:
        y0_train = a0_train + (b0_train * z)
        beta_target = transfer_learning_hdr.solve_lasso(X0_train, y0_train, lambda_sel)
        _, active_target, sign_target, _ = utils.construct_betaM_M_SM_Mc(beta_target)
        _, _, c0, d0, target_interval = target_fold_state_interval(
            X0_train,
            a0_train,
            b0_train,
            active_target,
            np.asarray(sign_target).ravel(),
            lambda_sel,
            len(train_indices),
        )

        y_aug_train = a_aug_train + (b_aug_train * z)
        beta_aug = transfer_learning_hdr.solve_lasso(X_aug_train, y_aug_train, lambda_sel)
        _, active_aug, sign_aug, _ = utils.construct_betaM_M_SM_Mc(beta_aug)
        _, _, ck, dk, aug_interval = augmented_fold_state_interval(
            X_aug_train,
            a_aug_train,
            b_aug_train,
            active_aug,
            np.asarray(sign_aug).ravel(),
            lambda_sel,
            X_aug_train.shape[0],
        )

        linear_region = utils.intersect_interval_unions(target_interval, aug_interval, tol=tol)
        linear_region = utils.clip_interval_union(linear_region, z, z_max, tol=tol)
        if not linear_region:
            z += eps
            continue

        Atilde, Btilde, Ctilde = validation_quadratic(X0_val, a0_val, b0_val, c0, d0, ck, dk, len(fold_indices))
        quad_region = utils.solve_quadratic_leq(Atilde, Btilde, Ctilde)
        local_region = utils.intersect_interval_unions(linear_region, quad_region, tol=tol)
        local_region = utils.clip_interval_union(local_region, z_min, z_max, tol=tol)
        intervals = utils.union_interval_unions(intervals, local_region, tol=tol)

        right_endpoint = linear_region[-1][1]
        if not np.isfinite(right_endpoint):
            break
        if right_endpoint <= z + tol:
            z += eps
        else:
            z = right_endpoint + eps

    return utils.clip_interval_union(intervals, z_min, z_max, tol=tol)


def source_selection_region(X0, Y0, XS_list, YS_list, a, b, folds, I_obs, lambda_sel, z_min, z_max, eps=1e-5, tol=1e-10):
    total_region = [(z_min, z_max)]
    majority = (len(folds) + 1) // 2
    selected_sources = set(I_obs)

    for source_idx in range(len(XS_list)):
        win_regions = [
            fold_win_region(
                source_idx,
                fold_idx,
                X0,
                Y0,
                XS_list,
                YS_list,
                a,
                b,
                folds,
                lambda_sel,
                z_min,
                z_max,
                eps=eps,
                tol=tol,
            )
            for fold_idx in range(len(folds))
        ]

        mode = "selected" if source_idx in selected_sources else "discarded"
        source_region = utils.count_region_from_fold_wins(win_regions, majority, mode, z_min=z_min, z_max=z_max, tol=tol)
        total_region = utils.intersect_interval_unions(total_region, source_region, tol=tol)
        if not total_region:
            return []

    return total_region


def model_selection_region(X0, Y0, XS_list, YS_list, a, b, I_obs, M_obs, lambda0, lambdak_list, z_min, z_max, eps=1e-5, tol=1e-10):
    ns_list = [ys.shape[0] for ys in YS_list]
    source_a_blocks, target_a = utils.split_stacked_response(a, ns_list, Y0.shape[0])
    source_b_blocks, target_b = utils.split_stacked_response(b, ns_list, Y0.shape[0])

    selected_source_set = list(I_obs)
    source_a_selected = [source_a_blocks[idx] for idx in selected_source_set]
    source_b_selected = [source_b_blocks[idx] for idx in selected_source_set]

    a_adapt = np.concatenate([
        np.asarray(source_a_selected[idx]).reshape(-1) / np.sqrt(source_a_selected[idx].shape[0])
        for idx in range(len(source_a_selected))
    ] + [target_a / np.sqrt(Y0.shape[0])])
    b_adapt = np.concatenate([
        np.asarray(source_b_selected[idx]).reshape(-1) / np.sqrt(source_b_selected[idx].shape[0])
        for idx in range(len(source_b_selected))
    ] + [target_b / np.sqrt(Y0.shape[0])])

    intervals = []
    z = z_min
    while z < z_max:
        Y0_z = target_a + (target_b * z)
        YS_z = [source_a_blocks[idx] + (source_b_blocks[idx] * z) for idx in range(len(XS_list))]
        theta_hat, beta0_hat, X_tilde, w_tilde = transfer_learning_hdr.solve_cort_model(
            X0,
            Y0_z,
            XS_list,
            YS_z,
            selected_source_set,
            lambda0,
            lambdak_list,
        )

        local_interval, target_active = kkt_interval(
            X_tilde,
            a_adapt,
            b_adapt,
            theta_hat,
            w_tilde,
            p=X0.shape[1],
            tol=tol,
        )

        local_interval = utils.clip_interval_union(local_interval, z, z_max, tol=tol)
        if not local_interval:
            z += eps
            continue

        if list(target_active) == list(M_obs):
            intervals = utils.union_interval_unions(intervals, local_interval, tol=tol)

        right_endpoint = local_interval[-1][1]
        if not np.isfinite(right_endpoint):
            break
        if right_endpoint <= z + tol:
            z += eps
        else:
            z = right_endpoint + eps

    return utils.clip_interval_union(intervals, z_min, z_max, tol=tol)


sub_prob = SimpleNamespace(
    lasso_state_interval=lasso_state_interval,
    target_fold_state_interval=target_fold_state_interval,
    augmented_fold_state_interval=augmented_fold_state_interval,
    validation_quadratic=validation_quadratic,
    kkt_interval=kkt_interval,
    fold_win_region=fold_win_region,
    source_selection_region=source_selection_region,
    model_selection_region=model_selection_region,
)

In [6]:
def validation_quadratic(X0_val, a0_val, b0_val, c0, d0, ck, dk, n_val):
    a0_val = np.asarray(a0_val, dtype=float).reshape(-1, 1)
    b0_val = np.asarray(b0_val, dtype=float).reshape(-1, 1)
    c0 = np.asarray(c0, dtype=float).reshape(-1, 1)
    d0 = np.asarray(d0, dtype=float).reshape(-1, 1)
    ck = np.asarray(ck, dtype=float).reshape(-1, 1)
    dk = np.asarray(dk, dtype=float).reshape(-1, 1)

    residual0_a = a0_val - X0_val @ c0
    residual0_b = b0_val - X0_val @ d0
    residualk_a = a0_val - X0_val @ ck
    residualk_b = b0_val - X0_val @ dk

    A0 = 0.5 * np.vdot(residual0_b, residual0_b).real / n_val
    B0 = np.vdot(residual0_a, residual0_b).real / n_val
    C0 = 0.5 * np.vdot(residual0_a, residual0_a).real / n_val

    Ak = 0.5 * np.vdot(residualk_b, residualk_b).real / n_val
    Bk = np.vdot(residualk_a, residualk_b).real / n_val
    Ck = 0.5 * np.vdot(residualk_a, residualk_a).real / n_val

    return float(Ak - A0), float(Bk - B0), float(Ck - C0)


def SI(X0, Y0, XS_list, YS_list, lambda_sel, lambda0, lambdak_list, SigmaS_list, Sigma0, folds=None, T=5, z_min=-20, z_max=20, verbose=False):
    if len(XS_list) != len(YS_list):
        raise ValueError("XS_list and YS_list must have the same length")
    if len(lambdak_list) != len(XS_list):
        raise ValueError("lambdak_list must have the same length as XS_list")

    if folds is None:
        folds = utils.construct_folds(X0.shape[0], T=T, shuffle=False)
    if len(folds) % 2 == 0:
        raise ValueError("The number of folds T must be odd")

    I_obs = transfer_learning_hdr.adaptive_source_selection(X0, Y0, XS_list, YS_list, folds, lambda_sel, verbose=verbose)
    _, beta0_hat, _, _ = transfer_learning_hdr.solve_cort_model(
        X0,
        Y0,
        XS_list,
        YS_list,
        I_obs,
        lambda0,
        lambdak_list,
        verbose=verbose,
        label="Observed CoRT fit",
    )
    M_obs = [idx for idx, value in enumerate(beta0_hat) if value != 0]

    if verbose:
        print(f"observed informative source set = {I_obs}")
        print(f"observed target active set = {M_obs}")

    if not M_obs:
        return None

    Y_obs = utils.construct_Y(YS_list, Y0)
    Sigma = utils.construct_Sigma(SigmaS_list, Sigma0)
    X0M = X0[:, M_obs]
    p_values = []
    for j in M_obs:
        etaj, etajTy = utils.construct_test_statistic(j, X0M, Y_obs, M_obs, Y0.shape[0], Y_obs.shape[0])
        a, b = utils.calculate_a_b(etaj, Y_obs, Sigma, Y_obs.shape[0])

        intervals_source = sub_prob.source_selection_region(X0, Y0, XS_list, YS_list, a, b, folds, I_obs, lambda_sel, z_min, z_max)
        intervals_model = sub_prob.model_selection_region(
            X0, Y0, XS_list, YS_list, a, b,
            I_obs, M_obs, lambda0, lambdak_list, z_min, z_max,
        )
        intervals = utils.intersect_interval_unions(intervals_source, intervals_model)
        p_sel = None
        if intervals:
            p_sel = utils.calculate_TN_p_value(intervals, etaj, etajTy, Sigma, 0.0)
        p_values.append((j, p_sel))

    return p_values


def SI_randj(X0, Y0, XS_list, YS_list, lambda_sel, lambda0, lambdak_list, SigmaS_list, Sigma0, folds=None, T=5, z_min=-20, z_max=20, verbose=False):
    if len(XS_list) != len(YS_list):
        raise ValueError("XS_list and YS_list must have the same length")
    if len(lambdak_list) != len(XS_list):
        raise ValueError("lambdak_list must have the same length as XS_list")

    if folds is None:
        folds = utils.construct_folds(X0.shape[0], T=T, shuffle=False)
    if len(folds) % 2 == 0:
        raise ValueError("The number of folds T must be odd")

    I_obs = transfer_learning_hdr.adaptive_source_selection(X0, Y0, XS_list, YS_list, folds, lambda_sel, verbose=verbose)
    _, beta0_hat, _, _ = transfer_learning_hdr.solve_cort_model(
        X0,
        Y0,
        XS_list,
        YS_list,
        I_obs,
        lambda0,
        lambdak_list,
        verbose=verbose,
        label="Observed CoRT fit",
    )
    M_obs = [idx for idx, value in enumerate(beta0_hat) if value != 0]

    if verbose:
        print(f"observed informative source set = {I_obs}")
        print(f"observed target active set = {M_obs}")

    if not M_obs:
        return None

    Y_obs = utils.construct_Y(YS_list, Y0)
    Sigma = utils.construct_Sigma(SigmaS_list, Sigma0)
    X0M = X0[:, M_obs]
    j = random.choice(M_obs)

    etaj, etajTy = utils.construct_test_statistic(j, X0M, Y_obs, M_obs, Y0.shape[0], Y_obs.shape[0])
    a, b = utils.calculate_a_b(etaj, Y_obs, Sigma, Y_obs.shape[0])

    intervals_source = sub_prob.source_selection_region(X0, Y0, XS_list, YS_list, a, b, folds, I_obs, lambda_sel, z_min, z_max)
    intervals_model = sub_prob.model_selection_region(
        X0, Y0, XS_list, YS_list, a, b,
        I_obs, M_obs, lambda0, lambdak_list, z_min, z_max,
    )
    intervals = utils.intersect_interval_unions(intervals_source, intervals_model)
    if not intervals:
        return None

    p_sel = utils.calculate_TN_p_value(intervals, etaj, etajTy, Sigma, 0.0)
    return p_sel


sub_prob.validation_quadratic = validation_quadratic

# Example Run

This last cell uses the standardized large demo setup and enables stage-wise active-set printing with `verbose=True`.

In [ ]:
np.random.seed(0)
random.seed(0)
verbose = True

XS_list, YS_list, X0, Y0, _, SigmaS_list, Sigma0, _ = gen_data.generate_data(
    p=300,
    s=5,
    nS=100,
    nT=50,
    true_beta=1.0,
    num_info_aux=1,
    num_uninfo_aux=1,
    gamma=0.01,
)

p_values = SI(
    X0,
    Y0,
    XS_list,
    YS_list,
    lambda_sel=0.5,
    lambda0=0.5,
    lambdak_list=[0.5] * len(XS_list),
    SigmaS_list=SigmaS_list,
    Sigma0=Sigma0,
    T=3,
    verbose=verbose,
)

print("Adaptive CoRT-SI p-values:")
print(p_values)

source 0, fold 1 target-only: active set = [0, 2, 3, 4, 112, 153, 160, 198, 235]
source 0, fold 1 target+source: active set = [1, 2, 3, 4]
source 0, fold 1: vote = True
source 0, fold 2 target-only: active set = [2, 3, 4, 31, 125, 127, 153, 210, 269]
source 0, fold 2 target+source: active set = [1, 2, 3, 4]
source 0, fold 2: vote = True
source 0, fold 3 target-only: active set = [2, 4, 65, 100, 149, 191, 205, 210, 261]
source 0, fold 3 target+source: active set = [1, 2, 3, 4]
source 0, fold 3: vote = True
source 0: votes = 3/3
source 1, fold 1 target-only: active set = [0, 2, 3, 4, 112, 153, 160, 198, 235]
source 1, fold 1 target+source: active set = [0, 1, 2, 3, 4, 82]
source 1, fold 1: vote = True
source 1, fold 2 target-only: active set = [2, 3, 4, 31, 125, 127, 153, 210, 269]
source 1, fold 2 target+source: active set = [0, 1, 2, 3, 4]
source 1, fold 2: vote = True
source 1, fold 3 target-only: active set = [2, 4, 65, 100, 149, 191, 205, 210, 261]
source 1, fold 3 target+source: ac